In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from tqdm import tqdm
import pandas as pd
import torch
import re

In [ ]:
model_name = "google/madlad400-7b-mt"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
input_file = "path/to/input.tsv"
df = pd.read_csv(input_file, sep='\t')

In [ ]:
def translate(texts, lang_prefix="<2it>"):
    input_texts = [f"{lang_prefix} Robot, {t}" for t in texts]

    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            length_penalty=1.0,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    translations = []
    for out in outputs:
        text = tokenizer.decode(out, skip_special_tokens=True)
        if text.startswith(lang_prefix):
            text = text[len(lang_prefix):].strip()
        try:
            text = text.removeprefix("Robot, ").lstrip()
            text = re.sub(r"[^\w\s']", '', text, flags=re.UNICODE).lower()
            text = re.sub(r"\se'\s", r" è ", text)
            text = re.sub(r"(\w)'\s*(\w)", r"\1' \2", text).strip()
        except:
            text = "TRANSLATION ERROR"
        translations.append(text)
    
    return translations

In [ ]:
def translate_dataframe(df, target_lang="<2it>", batch_size=10):
    results = []
    for i in tqdm(range(0, len(df), batch_size), desc="Processing batch"):
        batch_df = df.iloc[i:i+batch_size]
        batch_texts = batch_df['input'].astype(str).tolist()
        batch_ids = batch_df['id'].tolist()

        batch_translations = translate(batch_texts, target_lang)

        results.extend(list(zip(batch_ids, batch_texts, batch_translations)))
    
    return results

In [ ]:
translations = translate_dataframe(df)

In [ ]:
df_result = pd.DataFrame(translations, columns=["id", "input", "translation"])
output_file = "path/to/output/madlad400.tsv"
df_result.to_csv(output_file, sep='\t', index=False)
print(f"Translation completed. File saved in: {output_file}")